In [1]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy as sp
import scipy.signal as signal
import torchvision.transforms.functional as tvF

import math

In [2]:
table1 = np.zeros((6, 12))
table2 = np.zeros((6, 12))

In [3]:
def statistic(input, method):
    if method == 'mean':
        return input.mean()
    elif method == 'std':
        return input.std()


In [4]:
mean_interval = 6
std_interval = 3

In [5]:
def table_fun(save_dir, mean_interval, std_interval, table, test):
    '''
        classifier_ii + mean_interval * condition_ii: column 0, 1, 2; 6, 7, 8 for mean
        classifier_ii + mean_interval * condition_ii + std_interval: column 3, 4, 5, 9, 10, 11 for std
        row 5 for overall
        row 0 for STDP
        row 1 for BDP
        row 2 for BPTT1
        row 3 for BPTT2
        row 4 for NC
    '''
    for condition_ii, condition in enumerate(['fixedrepresentation', 'endtoend']):
        for classifier_ii, classifier in enumerate(['svm', 'TCNN', 'TTE']):
            if test == 'in':
                acu_dir = save_dir + f'/{classifier}_{condition}.npy'
            elif test == 'out':
                acu_dir = save_dir + f'/{classifier}_{condition}1.npy'
            acu_set = np.load(acu_dir)

            for value in ['mean', 'std']:
                if value == 'mean':
                    if classifier == 'svm':
                        table[-1, classifier_ii + mean_interval * condition_ii] = statistic(acu_set[:, 0], value)
                        for ii in range(1, 6):
                            if condition == 'fixedrepresentation':
                                table[ii - 1, classifier_ii + mean_interval * condition_ii] = statistic(acu_set[:, ii], value)
                            else:
                                if ii == 2:
                                    table[ii - 1, classifier_ii + mean_interval * condition_ii] = np.nan
                                elif ii == 1:
                                    table[ii - 1, classifier_ii + mean_interval * condition_ii] = statistic(acu_set[:, ii], value)
                                elif ii > 2:
                                    table[ii - 1, classifier_ii + mean_interval * condition_ii] = statistic(acu_set[:, ii - 1], value)

                    else:
                        table[-1, classifier_ii + mean_interval * condition_ii] = statistic(acu_set[:, 1], value)
                        for ii in range(1, 6):
                            if condition == 'fixedrepresentation':
                                table[ii - 1, classifier_ii + mean_interval * condition_ii] = statistic(acu_set[:, ii * 2 + 1], value)
                            else:
                                if ii == 2:
                                    table[ii - 1, classifier_ii + mean_interval * condition_ii] = np.nan
                                elif ii == 1:
                                    table[ii - 1, classifier_ii + mean_interval * condition_ii] = statistic(acu_set[:, ii * 2 + 1], value)
                                elif ii > 2:
                                    table[ii - 1, classifier_ii + mean_interval * condition_ii] = statistic(acu_set[:, (ii - 1) * 2 + 1], value)

                elif value == 'std':
                    if classifier == 'svm':
                        table[-1, classifier_ii + mean_interval * condition_ii + std_interval] = statistic(acu_set[:, 0], value)
                        for ii in range(1, 6):
                            if condition == 'fixedrepresentation':
                                table[ii - 1, classifier_ii + mean_interval * condition_ii + std_interval] = statistic(acu_set[:, ii], value)
                            else:
                                if ii == 2:
                                    table[ii - 1, classifier_ii + mean_interval * condition_ii + std_interval] = np.nan
                                elif ii == 1:
                                    table[ii - 1, classifier_ii + mean_interval * condition_ii + std_interval] = statistic(acu_set[:, ii], value)
                                elif ii > 2:
                                    table[ii - 1, classifier_ii + mean_interval * condition_ii + std_interval] = statistic(acu_set[:, ii - 1], value)

                    else:
                        table[-1, classifier_ii + mean_interval * condition_ii + std_interval] = statistic(acu_set[:, 1], value)
                        for ii in range(1, 6):
                            if condition == 'fixedrepresentation':
                                table[ii - 1, classifier_ii + mean_interval * condition_ii + std_interval] = statistic(acu_set[:, ii * 2 + 1], value)
                            else:
                                if ii == 2:
                                    table[ii - 1, classifier_ii + mean_interval * condition_ii + std_interval] = np.nan
                                elif ii == 1:
                                    table[ii - 1, classifier_ii + mean_interval * condition_ii + std_interval] = statistic(acu_set[:, ii * 2 + 1], value)
                                elif ii > 2:
                                    table[ii - 1, classifier_ii + mean_interval * condition_ii + std_interval] = statistic(acu_set[:, (ii - 1) * 2 + 1], value)

            del acu_set, acu_dir

    return table

In [6]:
save_dir = '/content/drive/MyDrive/Project/PlasticityDecoding/result'
table1 = table_fun(save_dir, mean_interval, std_interval, table1, 'in')
table2 = table_fun(save_dir, mean_interval, std_interval, table2, 'out')
table_dir = '/content/drive/MyDrive/Project/PlasticityDecoding/accuracy_table_1228'

In [7]:
type_list = ['STDP', 'BDP', 'BPTT1', 'BPTT2', 'NC', 'Overall']
with open(table_dir + '/table1.txt', 'w') as f:
    for ii in range(0, 6):
        for jj in range(0, 12):
            if jj == 0:
                f.write(type_list[ii] + ' ')
            if ii == 1 and jj > 5:
                f.write('&' + '{N/A}' + ' ')
            else:
                f.write('&' + str(np.round(table1[ii, jj], 3)) + ' ')
            if jj == 11:
                f.write('\\\\')
        f.write('\n')

In [8]:
type_list = ['STDP', 'BDP', 'BPTT1', 'BPTT2', 'NC', 'Overall']
with open(table_dir + '/table2.txt', 'w') as f:
    for ii in range(0, 6):
        for jj in range(0, 12):
            if jj == 0:
                f.write(type_list[ii] + ' ')
            if ii == 1 and jj > 5:
                f.write('&' + '{N/A}' + ' ')
            else:
                f.write('&' + str(np.round(table2[ii, jj], 3)) + ' ')
            if jj == 11:
                f.write('\\\\')
        f.write('\n')